# CASMI 2026 本地基线

先阅读 README.md 完成解压和依赖安装。此笔记本调用项目命令，不在 Notebook 里复制算法。数据默认在 `/root/autodl-tmp/data`。只有手动打开训练开关才会执行训练。此文件未执行。

In [ ]:
from pathlib import Path
import subprocess, sys

PROJECT = Path('/root/autodl-tmp/casmi26-baseline')  # 改为解压后的项目目录
assert (PROJECT / 'casmi_baseline').exists(), PROJECT
PYTHON = str(PROJECT / '.venv/bin/python') if (PROJECT / '.venv/bin/python').exists() else sys.executable

def run(*arguments):
    subprocess.check_call([PYTHON, '-m', 'casmi_baseline.cli', *map(str, arguments)], cwd=PROJECT)

run('doctor')
run('inspect')


## 1. 创建完整缓存（CPU 数据处理，不是模型训练）

首次运行一次；如果工作目录已有缓存，不要重复运行。更改清洗/指纹参数时，请在新工作目录重建。

In [ ]:
run('prepare')


## 2. 纯检索输出

这是最快的可提交基线，不需要训练。当前公开测试只是格式样例，不要用这里的结果评估泛化成绩。

In [ ]:
run('predict', '--output', PROJECT / 'submission.csv')
run('check-submission', '--path', PROJECT / 'submission.csv')


## 3. 无参考谱验证

按评分结构身份留出，并从全部来源屏蔽这些身份的参考谱。结构本身仍在候选池，模拟第二类已知结构。

In [ ]:
run('make-split', '--kind', 'identity', '--fraction', '0.1', '--molecules', '400')
run('validate', '--scenario', 'no-reference')


## 4. 外部候选（可选）

先用 `inspect-candidates --path ...` 查看 CSV/Parquet 的实际列名，再按 README 导入；不会自动获取社区数据库或权重。第二类分子的正确结构需先进入候选池，指纹模型才有可能将其排在前列。

In [ ]:
# run('inspect-candidates', '--path', '/root/autodl-tmp/data/community')
# run('import-candidates', '--path', '/path/to/candidates.parquet',
#     '--smiles-column', 'canonical_smiles', '--name', 'community')


## 5. 显式可选训练：CV 指纹模型

默认不开启。先安装支持 RTX 2080 Ti 的 PyTorch，再手动把 `RUN_TRAINING` 改成 True。batch=256 是起点，显存不足可用 128/64。模型前向验证没有替代真实训练；应观察损失和分子级 MRR。

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    run('train-fingerprint', '--split', PROJECT / 'work/validation/split.json',
        '--device', 'cuda', '--batch-size', '256', '--epochs', '12', '--steps-per-epoch', '1500')
else:
    print('未开启训练；如需训练，请手动设置 RUN_TRAINING=True。')


## 6. 评估 CV 权重

仅当完成上一格训练后再执行。全量模型会被验证命令拒绝，避免身份泄漏。

In [ ]:
CV_MODEL = PROJECT / 'work/models/cv/best.pt'
if CV_MODEL.exists():
    run('validate', '--scenario', 'no-reference', '--checkpoint', CV_MODEL, '--device', 'cuda')
else:
    print('还没有 CV 权重；先完成可选训练。')


## 7. 全量训练与本地推理（可选）

先根据 CV 结果确定 epoch 数。全量训练不计算验证成绩。默认训练开关仍关闭。

In [ ]:
RUN_FULL_TRAINING = False
SELECTED_EPOCHS = 12  # 按 CV 结果修改
if RUN_FULL_TRAINING:
    run('train-fingerprint', '--full', '--device', 'cuda', '--batch-size', '256',
        '--epochs', str(SELECTED_EPOCHS), '--steps-per-epoch', '1500')

FINAL_MODEL = PROJECT / 'work/models/full/final.pt'
if FINAL_MODEL.exists():
    run('predict', '--checkpoint', FINAL_MODEL, '--device', 'cuda', '--output', PROJECT / 'submission.csv')


## 8. 导出 Kaggle 资产

执行前确认完整缓存已生成，导出目录须为空。无权重时导出纯检索基线。将导出文件上传为 Kaggle Dataset，并用 `kaggle_submission.ipynb` 禁网运行。

In [ ]:
EXPORT_ASSETS = False
if EXPORT_ASSETS:
    command = ['export-assets', '--dest', '/root/autodl-tmp/casmi26-inference-assets']
    if FINAL_MODEL.exists():
        command += ['--checkpoint', FINAL_MODEL]
    run(*command)
